# 0. Configuration

In [26]:
# data load
from pathlib import Path # help with folder and file path 

# paths
import os 
# dataframe
import pandas as pd
import numpy as np
from functools import reduce

# data visualization
import matplotlib.pyplot as plt

# Preprocessing
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

# Cross Validation
from sklearn.model_selection import StratifiedKFold, cross_val_score

# Model
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

# Evaluation
from sklearn.metrics import roc_auc_score, precision_recall_curve


In [18]:
# build paths

base_dir = os.getcwd()

# join parts of the path

def get_pd(folder,name):
    path = os.path.join('datasets',folder,name)
    return path

# 1. Load Data

In [19]:
X_train_path = get_pd('train','X_train.csv')
X_train = pd.read_csv(X_train_path)
X_train = X_train.iloc[:,1:] # drop the first column which is just index
X_train

,latest_changed_password,max_item_count,days_since_first_transaction,email_address_age_days,email_domain_label,email_domain_tld_label,email_username_length,num_unique_delivery_addresses,latest_delivery_address_name_length,latest_delivery_address_region_label,...,per_week_payment_method_change,per_day_add_item_to_cart,per_day_transactions,per_day_payment_method_change,per_day_devices_per_user,per_day_purchase_total,per_day_unique_billing_last4,per_month_logout,per_month_page_activity,per_month_transactions
0,Missing,1.0,42.925145,140.053691,386.0,17.0,11.0,1.0,10.0,34.0,...,5.0,-999.0,4.0,5.0,1.0,23.98,3.0,-999.0,-999.0,2.505361
1,Missing,1.0,444.602262,134.242133,386.0,17.0,11.0,1.0,15.0,30.0,...,0.0,-999.0,-999.0,0.0,1.0,0.00,-999.0,-999.0,-999.0,4.483389
2,False,2.0,314.989689,4.757439,386.0,17.0,12.0,3.0,11.0,33.0,...,10.0,-999.0,5.0,7.0,2.0,0.00,4.0,-999.0,-999.0,12.503344
3,False,4.0,314.989689,22.485073,1100.0,17.0,9.0,1.0,11.0,33.0,...,2.0,-999.0,3.0,2.0,1.0,0.00,3.0,-999.0,-999.0,19.816075
4,False,2.0,314.989689,633.680181,64.0,17.0,6.0,1.0,11.0,24.0,...,0.0,-999.0,1.0,0.0,1.0,66.46,1.0,-999.0,-999.0,2.481141
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13183,Missing,3.0,6.373022,363.475348,386.0,17.0,9.0,1.0,12.0,34.0,...,0.0,-999.0,1.0,0.0,1.0,55.55,1.0,-999.0,-999.0,4.483389
13184,Missing,2.0,158.621297,2.193305,730.0,17.0,15.0,1.0,12.0,35.0,...,0.0,-999.0,-999.0,0.0,1.0,-999.00,-999.0,-999.0,1.0,4.101334
13185,False,-999.0,-999.000000,7.180110,386.0,17.0,15.0,-999.0,13.0,7.0,...,0.0,-999.0,-999.0,0.0,1.0,-999.00,-999.0,-999.0,-999.0,-999.000000
13186,Missing,1.0,1511.193006,871.043785,386.0,17.0,15.0,1.0,12.0,5.0,...,1.0,-999.0,3.0,1.0,1.0,33.14,2.0,-999.0,-999.0,11.629628


In [20]:
y_train_path = get_pd('train','y_train.csv')
y_train = pd.read_csv(y_train_path)
y_train = y_train.iloc[:,1:] # drop the first column which is just index
y_train

,is_fraudulent
0,0
1,0
2,1
3,1
4,0
...,...
13183,0
13184,0
13185,0
13186,0


In [34]:
# load test data
X_test_path = get_pd('test','X_test.csv')
X_test = pd.read_csv(X_test_path)
X_test = X_test.iloc[:,1:] # drop the first column which is just index
X_test

,latest_changed_password,max_item_count,days_since_first_transaction,email_address_age_days,email_domain_label,email_domain_tld_label,email_username_length,num_unique_delivery_addresses,latest_delivery_address_name_length,latest_delivery_address_region_label,...,per_week_payment_method_change,per_day_add_item_to_cart,per_day_transactions,per_day_payment_method_change,per_day_devices_per_user,per_day_purchase_total,per_day_unique_billing_last4,per_month_logout,per_month_page_activity,per_month_transactions
0,False,1.0,1237.847438,1533.666613,575.0,17.0,11.0,2.0,12.0,46.0,...,0.0,-999.0,-999.0,0.0,1.0,0.00,-999.0,-999.0,-999.0,12.074369
1,False,3.0,1207.080620,1066.933394,386.0,17.0,7.0,1.0,11.0,5.0,...,0.0,7.0,-999.0,0.0,2.0,0.00,1.0,-999.0,5.0,0.735138
2,Missing,2.0,537.175247,664.548801,386.0,17.0,7.0,1.0,13.0,10.0,...,0.0,-999.0,2.0,0.0,1.0,78.59,1.0,-999.0,-999.0,4.089693
3,False,1.0,317.478958,126.923077,386.0,17.0,13.0,2.0,11.0,18.0,...,1.0,-999.0,2.0,1.0,1.0,61.27,2.0,-999.0,-999.0,4.483389
4,Missing,-999.0,-999.000000,-999.000000,-999.0,-999.0,-999.0,-999.0,-999.0,-999.0,...,0.0,-999.0,-999.0,0.0,1.0,-999.00,-999.0,-999.0,-999.0,-999.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3292,False,1.0,441.495962,257.031888,1100.0,17.0,9.0,1.0,12.0,44.0,...,1.0,3.0,-999.0,0.0,2.0,0.00,-999.0,1.0,1.0,7.331166
3293,False,2.0,164.158927,1009.217491,64.0,17.0,14.0,1.0,13.0,10.0,...,0.0,-999.0,-999.0,0.0,1.0,0.00,-999.0,-999.0,-999.0,4.483389
3294,False,1.0,314.989689,649.766126,386.0,17.0,9.0,1.0,14.0,17.0,...,0.0,-999.0,3.0,0.0,1.0,67.55,-999.0,-999.0,-999.0,7.081968
3295,False,1.0,238.406580,46.255889,492.0,28.0,6.0,1.0,12.0,16.0,...,0.0,-999.0,-999.0,0.0,1.0,0.00,-999.0,-999.0,-999.0,2.225910


In [54]:
# load test labels
y_test = pd.read_csv(get_pd('test', 'y_test.csv'), index_col=0)["is_fraudulent"]
y_test

12780    0
14336    0
12951    0
13108    0
1360     0
        ..
6552     0
2684     0
15844    0
6120     0
12828    0
Name: is_fraudulent, Length: 3297, dtype: int64

In [ ]:
ids_train = pd.read_csv(get_pd('train', 'ids_train.csv'), index_col=0)["consumer_id"]


In [55]:
ids_test = pd.read_csv(get_pd('test', 'ids_test.csv'), index_col=0)["consumer_id"]
ids_test

12780    16008
14336    17953
12951    16218
13108    16411
1360      1697
         ...  
6552      8217
2684      3356
15844    19864
6120      7653
12828    16067
Name: consumer_id, Length: 3297, dtype: int64

# 2. Feature Engineering

Feature engineering is part of the model development pipeline and can also be part of model selection when different feature sets are compared using validation results. In this project, feature transformations that learn from the data, such as encoding, imputation, scaling, and resampling, must be applied within the cross-validation process to **avoid data leakage**. This ensures that each validation fold remains unseen during training and gives a more reliable estimate of model performance.


## 2a. K-Fold Cross Validation

After splitting the data into training and test sets, I set up a cross-validation (CV) loop on the training data only. Cross-validation is a resampling method used during model development to evaluate how well a model is likely to generalize to unseen data.

In this project, I use Stratified K-Fold cross-validation, which repeatedly splits the training set into several folds while preserving the original fraud ratio (~6.5%) in each fold. For each iteration, the model is trained on `K-1` folds and validated on the remaining fold. This process is repeated until every fold has been used once as a validation set, and the performance is then averaged across all folds.

This step is especially important for an imbalanced fraud dataset because a single validation split may give an unstable estimate of model performance. Stratified cross-validation provides a more reliable assessment while maintaining the minority fraud class proportion in each fold.

Cross-validation is only applied to the training set, not the test set. The test set is kept untouched until the very end so it can provide a final unbiased evaluation of the selected model.


In [21]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

## 2b. Preprocessor

Before fitting the model on training set, preprocessing categorical values and standardizing numerical values are important. For categorical values, there are several methods, but the key is to keep in mind about the data's cardinality.

My training data has:
- latest_changed_password: 3 unique values
- latest_item_category: about 3,109 unique values
- latest_item_product_title: about 6,906 unique values


Best simple baseline: one-hot low-cardinality + frequency encoding high-cardinality

Best stronger experiment: one-hot low-cardinality + target encoding high-cardinality

Best model-driven option: try CatBoost with native categorical handling


In [27]:

class FrequencyEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, normalize=True):
        self.normalize = normalize

    def fit(self, X, y=None):
        X = pd.DataFrame(X).copy()
        self.columns_ = X.columns.tolist()
        self.maps_ = {
            col: X[col].value_counts(normalize=self.normalize, dropna=False).to_dict()
            for col in self.columns_
        }
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        X.columns = self.columns_
        for col in self.columns_:
            X[col] = X[col].map(self.maps_[col]).fillna(0.0)
        return X.to_numpy()

    def get_feature_names_out(self, input_features=None):
        input_features = self.columns_ if input_features is None else input_features
        suffix = "freq" if self.normalize else "count"
        return np.array([f"{col}_{suffix}" for col in input_features], dtype=object)


In [28]:
ohe_cols = ["latest_changed_password"]
freq_cols = ["latest_item_category", "latest_item_product_title"]

numerical_features = X_train.select_dtypes(include=["float64","int64"]).columns.tolist()

preprocessor = ColumnTransformer(
    transformers = [
        # handle unknown category
        ("ohe_cols",OneHotEncoder(handle_unknown="ignore"),ohe_cols),
        ("freq_cols",FrequencyEncoder(normalize=True),freq_cols),
        ("num",StandardScaler(), numerical_features),
        
    ]
)


# 3. Baseline Model - Logistic Regression

In [29]:
pipeline = Pipeline([
    ("prep", preprocessor),
    ("model", LogisticRegression(class_weight="balanced", max_iter=1000))
])

scoring computes average prcision (AP) which tells how well the model ranks true fraud cases above non-fraud cases across many thresholds.


In [30]:
scores = cross_val_score(
    pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring="average_precision"
)

/Users/chohasong/anaconda3/lib/python3.11/site-packages/sklearn/utils/validation.py:1406: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/chohasong/anaconda3/lib/python3.11/site-packages/sklearn/utils/validation.py:1406: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/chohasong/anaconda3/lib/python3.11/site-packages/sklearn/utils/validation.py:1406: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/chohasong/anaconda3/lib/python3.11/site-packages/sklearn/utils/validation.py:1406: DataConversionWarning: A column-vector y was passed when a 1d array was e

In [33]:
print("Fold AP scores:", scores)
print("Mean AP:", scores.mean())

Fold AP scores: [0.61912468 0.68440478 0.65874132 0.68243472 0.65567818]
Mean AP: 0.6600767353124269


# 4. Fit baseline on full training set

In [46]:
from sklearn.metrics import average_precision_score, roc_auc_score, classification_report, confusion_matrix


In [37]:
# fit model on all training data
pipeline.fit(X_train, y_train)


/Users/chohasong/anaconda3/lib/python3.11/site-packages/sklearn/utils/validation.py:1406: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


,steps,"[('prep', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('ohe_cols', ...), ('freq_cols', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [ ]:
# score test set 
y_proba = pipeline.predict_proba(X_test)[:, 1]
y_pred = (y_proba >= 0.5).astype(int)

print("Test AP:", average_precision_score(y_test, y_proba))
print("Test ROC AUC:", roc_auc_score(y_test, y_proba))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))


Test AP: 0.6801924130815247
Test ROC AUC: 0.9601023400562573
[[2812  268]
 [  24  193]]
              precision    recall  f1-score   support

           0       0.99      0.91      0.95      3080
           1       0.42      0.89      0.57       217

    accuracy                           0.91      3297
   macro avg       0.71      0.90      0.76      3297
weighted avg       0.95      0.91      0.93      3297



# 5. Baseline Case Table

In [56]:
print(type(ids_test), ids_test.shape)
print(type(y_test), y_test.shape)
print(type(y_proba), y_proba.shape)
print(type(y_pred), y_pred.shape)


<class 'pandas.core.series.Series'> (3297,)
<class 'pandas.core.series.Series'> (3297,)
<class 'numpy.ndarray'> (3297,)
<class 'numpy.ndarray'> (3297,)


In [57]:
baseline_cases = pd.DataFrame({
    "consumer_id": ids_test.values,
    "y_true": y_test.values,
    "risk_score": y_proba,
    "predicted_label": y_pred
})
baseline_cases

,consumer_id,y_true,risk_score,predicted_label
0,16008,0,0.000320,0
1,17953,0,0.003519,0
2,16218,0,0.001908,0
3,16411,0,0.014112,0
4,1697,0,0.003370,0
...,...,...,...,...
3292,8217,0,0.966323,1
3293,3356,0,0.005665,0
3294,19864,0,0.019353,0
3295,7653,0,0.226455,0


In [59]:
baseline_cases["risk_band"] = pd.cut(
    baseline_cases["risk_score"],
    bins=[0, 0.3, 0.7, 1.0],
    labels=["low", "medium", "high"],
    include_lowest=True
)


In [ ]:
# 1. Attach the raw feature context
case_intel_df = pd.concat(
    [
        baseline_cases.reset_index(drop=True),
        X_test.reset_index(drop=True)
    ],
    axis=1
)


In [63]:
# 2. Add outcome / error flags
case_intel_df["is_tp"] = ((case_intel_df["predicted_label"] == 1) & (case_intel_df["y_true"] == 1)).astype(int)
case_intel_df["is_fp"] = ((case_intel_df["predicted_label"] == 1) & (case_intel_df["y_true"] == 0)).astype(int)
case_intel_df["is_fn"] = ((case_intel_df["predicted_label"] == 0) & (case_intel_df["y_true"] == 1)).astype(int)
case_intel_df["is_tn"] = ((case_intel_df["predicted_label"] == 0) & (case_intel_df["y_true"] == 0)).astype(int)


In [ ]:
# Create reason codes

high_txn_threshold = X_train["per_day_transactions"].quantile(0.95)
high_ip_threshold = X_train["per_week_unique_ips"].quantile(0.95)

case_intel_df["high_transaction_velocity"] = (
    case_intel_df["per_day_transactions"] >= high_txn_threshold
).astype(int)

case_intel_df["many_unique_ips"] = (
    case_intel_df["per_week_unique_ips"] >= high_ip_threshold
).astype(int)


In [ ]:
# 4. Map reason codes into root-cause categories
case_intel_df["reason_high_velocity"] = case_intel_df["high_transaction_velocity"] == 1
case_intel_df["reason_many_ips"] = case_intel_df["many_unique_ips"] == 1

In [66]:
# 5. Use risk_band for segment analysis
case_intel_df.groupby("risk_band").agg(
    cases=("consumer_id", "count"),
    fraud_rate=("y_true", "mean"),
    predicted_fraud_rate=("predicted_label", "mean"),
    avg_score=("risk_score", "mean")
)


/var/folders/pp/n1knqjg57ds10zbbb7v7yvfw0000gn/T/ipykernel_82838/2387014581.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  case_intel_df.groupby("risk_band").agg(


,cases,fraud_rate,predicted_fraud_rate,avg_score
risk_band,,,,
low,2631,0.005701,0.00000,0.045602
medium,309,0.074434,0.33657,0.450331
high,357,0.501401,1.00000,0.900681


# Save Data

In [ ]:
case_intel_df.to_csv("datasets/case_intelligence_baseline.csv", index=False)

# References

https://www.ibm.com/think/tutorials/gradient-boosting-classifier#179371088

https://xgboost.readthedocs.io/en/release_3.2.0/get_started.html


https://feature-engine.trainindata.com/en/1.8.x/user_guide/encoding/WoEEncoder.html
